<a href="https://colab.research.google.com/github/Alish7495/Retrieval-Augmented-Generation/blob/main/rag_company_regulations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*# Imagine a company that has hundreds of internal regulations — HR rules, employee benefits, leave policies, dress code, and so on.*
--------------------------------------------------------------------------------
*# Employees often need to look up these rules quickly, but searching long PDF documents or files can be time-consuming.*
--------------------------------------------------------------------------------
# To solve that, we can build a Retrieval-Augmented Generation (RAG) system that connects a language model to the company’s internal regulation database.
# The model can retrieve the most relevant policy sections and generate clear, conversational answers for employees — just like an intelligent HR assistant. *

## **[Step1: Install the libraries]**

In [ ]:
# @title
!pip install chromadb langchain langchain-openai langchain-community langserve fastapi uvicorn langsmith openai tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━

## [**Step 2: Mounting google drive in case if your data is stored there**]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = "/content/drive/MyDrive/company_rag"
os.makedirs(BASE_DIR, exist_ok=True)


Mounted at /content/drive


## **Step 3 : Access and import your API keys for Open AI and LangSmith: We are using openAI models for embedding and generation and the Use Langsmith for tracing our retrieval and generation**

In [ ]:
import os, getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# (Optional) LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")
os.environ["LANGCHAIN_PROJECT"] = "company-regulations-rag"


Enter your OpenAI API key: ··········
Enter your LangSmith API key: ··········


## **Step 4 : We create a sets of instruction for the company: The Regulations!**

In [ ]:
DATA_PATH = f"{BASE_DIR}/company_regulations.jsonl"

# Fetch the regulations dataset from this repo's data/ folder (works in Colab and locally)
import urllib.request

RAW_URL = "https://raw.githubusercontent.com/Alish7495/Retrieval-Augmented-Generation/main/data/company_regulations.jsonl"
urllib.request.urlretrieve(RAW_URL, DATA_PATH)

print("Saved:", DATA_PATH)
!wc -l "$DATA_PATH"


## Step 5 : Embedding and create the data vector store

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
from tqdm import tqdm
import chromadb
from openai import OpenAI
import os

# === PATHS ===
BASE_DIR = "/content/drive/MyDrive/company_rag"
CHROMA_DIR = BASE_DIR  # ✅ Parent directory, not the inner folder
DATA_PATH = "/content/drive/MyDrive/company_rag/company_regulations.jsonl"
COLLECTION = "company_regulations"
EMBED_MODEL = "text-embedding-3-large"

# Ensure folder exists
os.makedirs(BASE_DIR, exist_ok=True)

# === Setup Chroma and OpenAI ===
client = OpenAI()
chroma = chromadb.PersistentClient(path=CHROMA_DIR)

collection = chroma.get_or_create_collection(
    name=COLLECTION,
    metadata={"source": "internal_policy_docs"}
)

# === Load and embed dataset ===
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

texts = [item["content"] for item in data]
metas = [{"topic": item["topic"], "id": item["id"]} for item in data]

print(f"Embedding {len(texts)} regulations...")

embs = [e.embedding for e in client.embeddings.create(model=EMBED_MODEL, input=texts).data]

# === Add to Chroma ===
for i, emb in enumerate(tqdm(embs)):
    collection.add(
        ids=[data[i]["id"]],
        documents=[texts[i]],
        metadatas=[metas[i]],
        embeddings=[emb]
    )

print(f"✅ Indexed {len(texts)} regulations into {CHROMA_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Embedding 100 regulations...


100%|██████████| 100/100 [00:07<00:00, 14.22it/s]

✅ Indexed 100 regulations into /content/drive/MyDrive/company_rag


## **Step 6 : Setup the Retriever generator and connecting to the database (we also set our prompt here)**

In [ ]:
# --- 🔧 Setup Cell (run only once) ---
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

CHROMA_DIR = "/content/drive/MyDrive/company_rag"

# Load vector DB
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectordb = Chroma(
    persist_directory=CHROMA_DIR,
    collection_name="company_regulations",
    embedding_function=embeddings
)
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

# Load LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Create Prompt
template = """
You are an HR assistant helping employees understand company regulations.
Always answer clearly using information from the retrieved context.

If the answer is not in the provided context, say:
"I’m not sure about this policy. Please check with the HR department."

Context:
{context}

Question:
{question}

Answer:
"""
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# Create the chain (reusable)
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)

print("✅ RAG system initialized — ready for questions!")


✅ RAG system initialized — ready for questions!


## **Step 7 : Ready to test! Then we can trace and check the retrieval and genewration in LangChain platform)**

In [ ]:
# --- 💬 Question Cell (re-run freely) ---
query = "What benefits are provided to employees after completing the probation period?"
result = rag_chain.invoke({"query": query})

print("🧠 AI Answer:\n", result["result"])


🧠 AI Answer:
 I’m not sure about this policy. Please check with the HR department.


**Examples!**

What is the company’s policy on remote work or working from home?

How many days of annual leave are employees entitled to each year?

Are employees allowed to take unpaid leave, and under what conditions?

What are the company’s rules regarding overtime and compensation for extra hours?

What is the procedure for reporting workplace harassment or discrimination?

Can employees use their personal devices for official work (BYOD policy)?

How is performance evaluation conducted, and how often does it occur?

What is the dress code policy for regular workdays and formal events?

What benefits are provided to employees after completing the probation period?

How should employees request approval for attending external training or conferences?

## **## More Queries!**

In [ ]:
# --- 🧠 Batch Test Cell ---
questions = [
    "What is the company’s policy on remote work or working from home?",
    "How many days of annual leave are employees entitled to each year?",
    "Are employees allowed to take unpaid leave, and under what conditions?",
    "What are the company’s rules regarding overtime and compensation for extra hours?",
    "What is the procedure for reporting workplace harassment or discrimination?",
    "Can employees use their personal devices for official work (BYOD policy)?",
    "How is performance evaluation conducted, and how often does it occur?",
    "What is the dress code policy for regular workdays and formal events?",
    "What benefits are provided to employees after completing the probation period?",
    "How should employees request approval for attending external training or conferences?"
]

results = []

for q in questions:
    res = rag_chain.invoke({"query": q})
    results.append((q, res["result"]))

# --- 🪄 Print all answers nicely ---
for i, (q, a) in enumerate(results, start=1):
    print(f"\n{'='*70}")
    print(f"❓ Question {i}: {q}")
    print(f"🧠 Answer: {a}")



❓ Question 1: What is the company’s policy on remote work or working from home?
🧠 Answer: The company allows remote work only with manager approval. Eligibility for remote work is based on job function, performance, and business needs. Additionally, remote schedules must be agreed upon in writing.

❓ Question 2: How many days of annual leave are employees entitled to each year?
🧠 Answer: I’m not sure about this policy. Please check with the HR department.

❓ Question 3: Are employees allowed to take unpaid leave, and under what conditions?
🧠 Answer: I’m not sure about this policy. Please check with the HR department.

❓ Question 4: What are the company’s rules regarding overtime and compensation for extra hours?
🧠 Answer: Overtime must be approved in advance by a manager. Non-exempt employees will be compensated according to local wage laws for approved overtime hours.

❓ Question 5: What is the procedure for reporting workplace harassment or discrimination?
🧠 Answer: The procedure fo